In [22]:
import pandas as pd
import glob
import pickle

In [36]:
path = glob.glob('../../results/holmes/fuse-org-negation/facebook__bart-base/full/NONE/4048/**/0/done/preds.csv')
path2 = glob.glob('../../results/holmes/fuse-org-negation/microsoft__deberta-v3-base//full/NONE/4048/**/0/done/preds.csv')

In [37]:
files = pd.concat([pd.read_csv(file) for file in path])
files2 = pd.concat([pd.read_csv(file) for file in path2])

In [38]:
average_scores = files2.groupby('Unnamed: 0')['pred'].mean().reset_index()

The predictions from the dataframe under here should go into the app, since they are the averages for every sentence for fuse-org-negation, but for one ml model only

In [ ]:
average_scores

In [ ]:
loaded_dump = pd.read_pickle('../../dumps/holmes/facebook__bart-base__full__NONE__probe-fuse-org-negation__4048__False.pickle')
loaded_dump[0]

In [1]:
import sys

import pandas as pd
import glob
import pickle


sys.path.append('./src/')

In [2]:
from utils import data_loading

In [3]:
from defs.control_task_types import CONTROL_TASK_TYPES
from defs.probe_task_types import PROBE_TASK_TYPES

In [4]:
base_model = data_loading.load_model("facebook/bart-base", CONTROL_TASK_TYPES.NONE, "full")

/opt/miniconda3/envs/HolmesEvaluation/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [27]:
probe_frame = data_loading.load_probe_file('./data/holmes/fuse-org-negation/modified_samples.csv', CONTROL_TASK_TYPES.NONE)

In [28]:
probing_frames = data_loading.load_folds(
        probe_frame=probe_frame,
        base_model=base_model,
        probe_task_type= PROBE_TASK_TYPES.SENTENCE,
        encoding="full",
        encoding_batch_size=10,
    )


  0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  6.52it/s]


In [29]:
probing_frames

[{-1:                           inputs context  topic  org_label set-0  id  label  \
  0        (This a test sentence,)            NaN          0  test   0      0   
  1            (Do not reproduce,)            NaN          0  test   1      0   
  2  (But try this cool tool out,)            NaN          0  test   2      0   
  
                                        inputs_encoded  
  0  [[1.022, 0.2328, -0.6646, 0.629, 0.2129, 0.944...  
  1  [[0.5264, 0.3528, -1.194, 3.096, 0.2017, 1.424...  
  2  [[0.535, -0.1228, 0.2186, -0.4963, -0.1423, 0....  }]

In [30]:
loaded_probing_frames = data_loading.load_probing_frames(probing_frames, "full")

3it [00:00, 5758.77it/s]


In [31]:
loaded_probing_frames

[{'train': Empty DataFrame
  Columns: [inputs, context, topic, org_label, set-0, id, label, inputs_encoded, unique_inputs]
  Index: [],
  'dev': Empty DataFrame
  Columns: [inputs, context, topic, org_label, set-0, id, label, inputs_encoded, unique_inputs]
  Index: [],
  'test':                           inputs context  topic  org_label set-0  id  label  \
  0        (This a test sentence,)            NaN          0  test   0      0   
  1            (Do not reproduce,)            NaN          0  test   1      0   
  2  (But try this cool tool out,)            NaN          0  test   2      0   
  
                                        inputs_encoded unique_inputs  
  0  [1.022, 0.2328, -0.6646, 0.629, 0.2129, 0.944,...          (t,)  
  1  [0.5264, 0.3528, -1.194, 3.096, 0.2017, 1.424,...          (d,)  
  2  [0.535, -0.1228, 0.2186, -0.4963, -0.1423, 0.5...          (b,)  }]

In [32]:
input_dim = loaded_probing_frames[0]["test"].iloc[0]["inputs_encoded"].shape[-1]

In [107]:
from torch.optim import Adam
cfg = {'learning_rate': 0.001,
  'num_labels': 2,
  'input_dim': input_dim,                           
  'batch_size': 16,
  'optimizer': Adam,
  'hidden_dim': 0,
  'dropout': 0.2,
  'warmup_rate': 0.1,
  'num_hidden_layers': 0,
  'seed': 0} # put seeds 0,1,2,3,4,5 here 

In [108]:
from model.probing_model import LinearProbingModel

probing_model = LinearProbingModel.load_from_checkpoint('./results/holmes/fuse-org-negation/facebook__bart-base/full/NONE/4048/0/0/done/epoch=0-step=17.ckpt', hyperparameter=cfg)

In [109]:
# from model.probing_model import LinearProbingModel

# probing_model = LinearProbingModel(hyperparameter=cfg)

In [110]:
data_lolz = loaded_probing_frames[0]["test"]

In [111]:
from utils.data_loading import load_dataset

test_dataset = load_dataset(data_lolz)

In [112]:
custom_dataloader = probing_model.get_test_dataloader(test_dataset, 300, shuffle=False)

In [113]:
from pytorch_lightning import Trainer

trainer = Trainer(accelerator="auto", devices=1, precision='32')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [114]:
predictions = trainer.test(probing_model, dataloaders=[custom_dataloader])

/opt/miniconda3/envs/HolmesEvaluation/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=10` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

/opt/miniconda3/envs/HolmesEvaluation/lib/python3.10/site-packages/torch/nn/modules/module.py:1518: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       full test acc       │            0.0            │
│       full test f1        │            0.0            │
│      unseen test acc      │            0.0            │
│      unseen test f1       │            0.0            │
└───────────────────────────┴───────────────────────────┘

In [115]:
predictions

[{'full test acc': 0.0,
  'full test f1': 0.0,
  'unseen test acc': 0.0,
  'unseen test f1': 0.0}]

In [116]:
test_predictions = [
            (
                instance_input,
                pred,
                instance_label,
                loss,
            )
            for instance_input, instance_label, pred, loss in zip(
                test_dataset.inputs,
                test_dataset.labels,
                probing_model.test_preds,
                probing_model.test_losses,
            )
        ]

In [117]:
test_prediction_frame = pd.DataFrame(test_predictions)
test_prediction_frame.columns = ["instance", "pred", "label", "loss"]

In [118]:
test_prediction_frame[['instance', 'pred', 'loss']]

,instance,pred,loss
0,"(This a test sentence,)",1.0,0.929163
1,"(Do not reproduce,)",1.0,1.045433
2,"(But try this cool tool out,)",1.0,1.012169


---

In [23]:
from 'extension/backend/backend.py' import Backend

SyntaxError: invalid syntax (1101150212.py, line 1)

In [24]:
import sys
sys.path.append('./extension/backend/')

In [25]:
import backend

In [26]:
results = backend.Backend()

/opt/miniconda3/envs/HolmesEvaluation/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: '../../data/holmes/fuse-org-negation/modified_samples.csv'

In [17]:
results

,instance,pred,label,loss
0,(This is a sentence to check if test sentences...,1.0,0,0.993731
1,"(Another not so good sentence.,)",1.0,1,0.521494
2,"(This one is better,)",1.0,0,1.091259
3,"(This one might not be as good,)",1.0,1,0.412736
4,"(Theres no way to not be able to check this,)",1.0,0,1.058523
